In [8]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt


# ----------------------------
# Config
# ----------------------------
project_root = Path().resolve().parent
DATA_DIR = project_root / "data"
REPORT_DIR = project_root / "reports"
FIG_DIR = REPORT_DIR / "figures"

ORDERS_CSV = DATA_DIR / "olist_orders_dataset.csv"
OUT_MONTHLY_KPI = DATA_DIR / "monthly_kpi.csv"


# ----------------------------
# Helpers
# ----------------------------
def ensure_dirs() -> None:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)


def load_orders(path: Path) -> pd.DataFrame:
# Parse important datetime columns in ONE read
    date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    ]
    return pd.read_csv(path, parse_dates=date_cols)


def build_delivered_table(orders: pd.DataFrame) -> pd.DataFrame:
    delivered = orders.loc[orders["order_status"] == "delivered"].copy()

# Delivery days: delivered_customer - purchase
    delivered["delivery_days"] = (
    delivered["order_delivered_customer_date"] - delivered["order_purchase_timestamp"]
    ).dt.days

# Delivery delay: delivered_customer - estimated_delivery
    delivered["delivery_delay"] = (
    delivered["order_delivered_customer_date"] - delivered["order_estimated_delivery_date"]
    ).dt.days

# Safe guards: drop rows where dates missing or negative weird values
    delivered = delivered.dropna(subset=["delivery_days", "delivery_delay"])
    return delivered

orders = load_orders(ORDERS_CSV)
delivered = build_delivered_table(orders)
delivered["is_delayed"] = delivered["delivery_delay"] >0
delivered["is_delayed"].value_counts()
delay_rate = delivered["is_delayed"].mean()
delay_rate.round(5)

np.float64(0.06773)